# Fixed Data Builder for Drone Delivery
Create fully reproducible instances from the base coordinate file without modifying the original data.

In [13]:
from __future__ import annotations
import json
import math
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd
import numpy as np

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'source' / 'Last_Mile_Delivery_Coordinates.csv').exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
BASE_CSV = PROJECT_ROOT / 'data' / 'source' / 'Last_Mile_Delivery_Coordinates.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'notebook_generated'

In [14]:
def load_base_data(csv_path: Path | None = None) -> pd.DataFrame:
    csv_path = BASE_CSV if csv_path is None else Path(csv_path)
    df = pd.read_csv(csv_path)
    # Standardize columns to lower snake case if needed
    df = df.rename(columns={
        'District': 'district',
        'Road_Slot': 'road_slot',
        'Latitude': 'lat',
        'Longitude': 'lon'
    })
    return df

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    # Great-circle distance
    r = 6371.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def compute_distance_matrix(df: pd.DataFrame) -> pd.DataFrame:
    coords = df[['lat', 'lon']].to_numpy()
    n = len(coords)
    dist = np.zeros((n, n))
    for i in range(n):
        lat1, lon1 = coords[i]
        for j in range(i + 1, n):
            lat2, lon2 = coords[j]
            d = haversine_km(lat1, lon1, lat2, lon2)
            dist[i, j] = d
            dist[j, i] = d
    return pd.DataFrame(dist)

In [15]:
@dataclass
class InstanceConfig:
    # Instance splitting
    split_strategy: str = 'by_district_slot'  # 'by_district_slot', 'fixed_size'
    n_instances: int = 10
    fixed_size: int = 25
    min_instance_size: int = 10

    # Depot
    depot_method: str = 'centroid_instance'  # 'centroid_instance', 'centroid_all', 'fixed_coord'
    depot_coord: Optional[Tuple[float, float]] = None

    # Drones
    drones_mode: str = 'per_customers'  # 'fixed', 'per_customers'
    fixed_drones: int = 5
    customers_per_drone: int = 10
    min_drones: int = 3

    # Demand / payload
    demand_mode: str = 'seeded_random'  # 'uniform', 'seeded_random', 'distance_band'
    demand_seed: int = 42
    demand_min: float = 0.5
    demand_max: float = 2.0
    uniform_demand: float = 1.0

    # Energy model parameters
    energy_per_km: float = 1.0
    payload_factor: float = 0.2

    # Drone limits
    max_payload: float = 5.0
    max_battery_km: float = 25.0

    # No-fly zones
    no_fly_mode: str = 'none'  # 'none', 'seeded_circles'
    no_fly_seed: int = 7
    no_fly_count: int = 2
    no_fly_radius_km: float = 0.7

In [16]:
def split_instances(df: pd.DataFrame, cfg: InstanceConfig) -> List[pd.DataFrame]:
    if cfg.split_strategy == 'by_district_slot':
        groups = list(df.groupby(['district', 'road_slot']))
        groups = [g for _, g in groups if len(g) >= cfg.min_instance_size]
        if len(groups) >= cfg.n_instances:
            return [g.reset_index(drop=True) for g in groups[:cfg.n_instances]]
    # Fallback to fixed size
    chunks = []
    df_sorted = df.sort_values(['district', 'road_slot', 'lat', 'lon']).reset_index(drop=True)
    for i in range(0, len(df_sorted), cfg.fixed_size):
        chunk = df_sorted.iloc[i:i + cfg.fixed_size]
        if len(chunk) >= cfg.min_instance_size:
            chunks.append(chunk.reset_index(drop=True))
        if len(chunks) >= cfg.n_instances:
            break
    return chunks

def choose_depot(df: pd.DataFrame, cfg: InstanceConfig, all_df: pd.DataFrame) -> Tuple[float, float]:
    if cfg.depot_method == 'fixed_coord' and cfg.depot_coord:
        return cfg.depot_coord
    if cfg.depot_method == 'centroid_all':
        return float(all_df['lat'].mean()), float(all_df['lon'].mean())
    # Default: centroid of instance
    return float(df['lat'].mean()), float(df['lon'].mean())

def assign_demands(df: pd.DataFrame, cfg: InstanceConfig, depot: Tuple[float, float]) -> pd.Series:
    if cfg.demand_mode == 'uniform':
        return pd.Series([cfg.uniform_demand] * len(df))
    if cfg.demand_mode == 'distance_band':
        # Deterministic: farther customers get higher demand
        distances = df.apply(lambda r: haversine_km(depot[0], depot[1], r['lat'], r['lon']), axis=1)
        scaled = (distances - distances.min()) / (distances.max() - distances.min() + 1e-9)
        return cfg.demand_min + scaled * (cfg.demand_max - cfg.demand_min)
    # Default: seeded random
    rng = np.random.default_rng(cfg.demand_seed)
    return pd.Series(rng.uniform(cfg.demand_min, cfg.demand_max, size=len(df)))

def build_no_fly_zones(df: pd.DataFrame, cfg: InstanceConfig) -> List[Dict]:
    if cfg.no_fly_mode == 'none':
        return []
    rng = np.random.default_rng(cfg.no_fly_seed)
    lat_min, lat_max = df['lat'].min(), df['lat'].max()
    lon_min, lon_max = df['lon'].min(), df['lon'].max()
    zones = []
    for i in range(cfg.no_fly_count):
        zones.append({
            'type': 'circle',
            'id': f'NFZ-{i+1}',
            'center': [float(rng.uniform(lat_min, lat_max)), float(rng.uniform(lon_min, lon_max))],
            'radius_km': float(cfg.no_fly_radius_km)
        })
    return zones

In [17]:
def derive_drone_count(n_customers: int, cfg: InstanceConfig) -> int:
    if cfg.drones_mode == 'fixed':
        return cfg.fixed_drones
    return max(cfg.min_drones, math.ceil(n_customers / cfg.customers_per_drone))

def build_instance(df: pd.DataFrame, instance_id: int, cfg: InstanceConfig, all_df: pd.DataFrame) -> Dict:
    depot = choose_depot(df, cfg, all_df)
    demands = assign_demands(df, cfg, depot)
    no_fly = build_no_fly_zones(df, cfg)

    customers = df.copy()
    customers['customer_id'] = [f'C{instance_id:02d}_{i+1:03d}' for i in range(len(customers))]
    customers['demand'] = demands

    instance = {
        'instance_id': instance_id,
        'depot': {'lat': depot[0], 'lon': depot[1]},
        'n_drones': derive_drone_count(len(customers), cfg),
        'limits': {
            'max_payload': cfg.max_payload,
            'max_battery_km': cfg.max_battery_km
        },
        'energy_model': {
            'energy_per_km': cfg.energy_per_km,
            'payload_factor': cfg.payload_factor
        },
        'no_fly_zones': no_fly,
        'customers': customers
    }
    return instance

def save_instance(instance: Dict, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    customers = instance['customers']
    customers.to_csv(out_dir / 'customers.csv', index=False)

    dist = compute_distance_matrix(customers)
    dist.to_csv(out_dir / 'distance_km.csv', index=False)

    meta = {
        'instance_id': instance['instance_id'],
        'depot': instance['depot'],
        'n_drones': instance['n_drones'],
        'limits': instance['limits'],
        'energy_model': instance['energy_model'],
        'no_fly_zones': instance['no_fly_zones']
    }
    with open(out_dir / 'meta.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)

def generate_instances(cfg: InstanceConfig, csv_path: Path | None = None, out_root: Path | None = None) -> Path:
    csv_path = BASE_CSV if csv_path is None else Path(csv_path)
    out_root = OUTPUT_ROOT if out_root is None else Path(out_root)
    df = load_base_data(csv_path)
    instances = split_instances(df, cfg)

    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = out_root / f'run_{run_id}'
    run_dir.mkdir(parents=True, exist_ok=True)

    # Save the config snapshot for reproducibility
    with open(run_dir / 'config.json', 'w', encoding='utf-8') as f:
        json.dump(asdict(cfg), f, indent=2)

    for i, inst_df in enumerate(instances, start=1):
        instance = build_instance(inst_df, i, cfg, df)
        save_instance(instance, run_dir / f'instance_{i:02d}')

    return run_dir

## Example usage
Create a new run folder with 10 fixed instances (edit the config as needed).

In [18]:
cfg = InstanceConfig()
cfg.demand_mode = 'seeded_random'
cfg.no_fly_mode = 'seeded_circles'
run_dir = generate_instances(cfg)
print('Generated in:', run_dir)

Generated in: C:\Users\moham\Documents\Drone\data\notebook_generated\run_20260531_164130
